In [0]:
%run "../0_common/01. env-config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
circuits_df = spark.table(bronze_table)
display(circuits_df.head(5))

In [0]:
circuits_df = circuits_df.select("circuitId","circuitName","country","lat","long","locality","ingestion_timestamp","source_file")

In [0]:
circuits_df = circuits_df.withColumnsRenamed(
    {
        "circuitId":"circuit_id",
        "circuitName":"circuit_name",
        "lat":"latitude",
        "long":"longitude"
    }
)

In [0]:
import pyspark.sql.functions as F
circuits_df = circuits_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
circuits_df = circuits_df.dropDuplicates(["circuit_id"])
display(circuits_df)

In [0]:
circuits_df = (
    circuits_df
    .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
    .withColumn("locality", F.initcap(F.col("locality")))
)

In [0]:
display(circuits_df.head(5))

In [0]:
circuits_df.write.format('delta').mode("overwrite").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))